In [ ]:
# ── MLOps bootstrap (auto-injected by inject_mlops_cell.py) ──────────────────
import os, warnings, mlflow
warnings.filterwarnings("ignore")

SEED = 42
import random, numpy as np
random.seed(SEED)
np.random.seed(SEED)
try:
    import torch; torch.manual_seed(SEED)
except ImportError:
    pass
try:
    import tensorflow as tf; tf.random.set_seed(SEED)
except ImportError:
    pass

_nb_name = os.path.basename(os.path.abspath("__file__") if "__file__" in dir() else ".").replace(".ipynb","")
mlflow.set_tracking_uri("sqlite:///" + str(Path(__file__).parent.parent.parent / "mlflow.db")
                        if "__file__" in dir() else "sqlite:///mlflow.db")
_exp = mlflow.set_experiment(_nb_name or "unnamed_notebook")
print(f"MLflow experiment: {_exp.name}")


# 📊 Table + Text RAG Assistant

## What We're Building

A QA system that handles **both tabular data (CSVs/DataFrames) and text documents**
in a unified retrieval pipeline. The system detects whether a question needs
tabular reasoning or text lookup, routes to the right handler, and combines evidence.

## The Problem

```
Standard RAG:  "What was revenue in Q3?"  → Searches text chunks → Misses the table
Table + Text:  "What was revenue in Q3?"  → Searches table → Finds exact number
               "Explain the revenue trend" → Searches text  → Gets narrative context
```

## Architecture

```
User Question
     │
     ▼
┌──────────────┐
│  Router       │  → Is this a table or text question?
└──────┬───────┘
       ├──── table → Pandas query on DataFrame
       ├──── text  → ChromaDB vector search
       └──── both  → Query both, merge evidence
            │
            ▼
       LLM Answer
```

## Stack
- **pandas** — tabular data handling
- **LangChain + ChromaDB** — text retrieval
- **Ollama** — local LLM for routing + answering

## Step 1 — Imports & Create Sample Data

In [ ]:
import pandas as pd
from langchain_ollama import ChatOllama, OllamaEmbeddings
from langchain_community.vectorstores import Chroma
from langchain.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain.schema import Document

print("All imports successful!")

In [ ]:
# --- Tabular Data: Company Financial Metrics ---
financial_data = pd.DataFrame({
    "quarter": ["Q1 2023", "Q2 2023", "Q3 2023", "Q4 2023", "Q1 2024", "Q2 2024"],
    "revenue_millions": [45.2, 52.8, 58.1, 67.3, 71.5, 78.9],
    "expenses_millions": [38.1, 41.5, 44.2, 48.7, 50.3, 54.1],
    "employees": [320, 345, 372, 401, 425, 448],
    "new_customers": [150, 178, 203, 245, 267, 298],
    "churn_rate_pct": [3.2, 2.8, 2.5, 2.1, 1.9, 1.7],
    "nps_score": [42, 45, 48, 52, 55, 58],
})

product_data = pd.DataFrame({
    "product": ["CloudSync Pro", "DataVault", "API Gateway", "Analytics Hub", "DevOps Suite"],
    "launch_year": [2020, 2021, 2022, 2023, 2024],
    "monthly_active_users": [12500, 8700, 15200, 4300, 1800],
    "monthly_revenue_k": [320, 180, 410, 95, 45],
    "satisfaction_score": [4.5, 4.2, 4.7, 4.1, 4.8],
})

print("=== Financial Data ===")
print(financial_data.to_string(index=False))
print("\n=== Product Data ===")
print(product_data.to_string(index=False))

In [ ]:
# --- Text Documents: Company Reports & Analysis ---
text_docs = [
    Document(page_content="TechNova's Q3 2023 was a breakthrough quarter. Revenue crossed $58M for the first time, driven by strong enterprise adoption of API Gateway. The product team shipped 47 features, the most in any quarter. CEO Sarah Chen called it 'our best execution quarter ever'.", metadata={"source": "Q3 report"}),
    Document(page_content="The company's growth strategy centers on three pillars: (1) expanding enterprise accounts with multi-product bundles, (2) reducing churn through a dedicated customer success team launched in Q2 2023, and (3) international expansion starting with Europe in 2024.", metadata={"source": "strategy doc"}),
    Document(page_content="TechNova's declining churn rate from 3.2% to under 2% over six quarters is attributed to the new customer success program. Each enterprise customer now has a dedicated CSM (Customer Success Manager) who monitors usage patterns and intervenes proactively.", metadata={"source": "retention analysis"}),
    Document(page_content="API Gateway has become TechNova's fastest-growing product, reaching 15,200 MAU within two years of launch. Its success is driven by the developer-friendly documentation, generous free tier (10K requests/month), and seamless integration with CloudSync Pro.", metadata={"source": "product review"}),
    Document(page_content="The engineering team grew from 120 to 200 engineers in 2023. Key hires included VP of Engineering Maria Santos (ex-Stripe) and Head of ML James Park (ex-Google). The team adopted a platform engineering model, creating shared services used across all products.", metadata={"source": "eng report"}),
    Document(page_content="TechNova's NPS (Net Promoter Score) improvement from 42 to 58 over six quarters places it in the 'excellent' category for B2B SaaS. The main drivers were faster support response times (from 4h to 45min) and the self-service analytics dashboard.", metadata={"source": "CX report"}),
    Document(page_content="DevOps Suite, launched in early 2024, targets the CI/CD market. Despite having only 1800 MAU, early feedback is extremely positive (4.8 satisfaction). The product integrates with GitHub, GitLab, and Azure DevOps for one-click deployment pipelines.", metadata={"source": "product launch"}),
    Document(page_content="TechNova's revenue per employee improved from $141K in Q1 2023 to $176K in Q2 2024, indicating improving operational efficiency. Margins expanded as the cloud infrastructure team optimized costs by migrating to reserved instances and implementing auto-scaling.", metadata={"source": "ops report"}),
]

embeddings = OllamaEmbeddings(model="nomic-embed-text-v2-moe")
vectorstore = Chroma.from_documents(text_docs, embeddings, collection_name="table_text")

llm = ChatOllama(model="qwen3.5:9b", temperature=0.1)

print(f"\nIndexed {len(text_docs)} text documents in ChromaDB")
print(f"Available tables: financial_data ({len(financial_data)} rows), product_data ({len(product_data)} rows)")

## Step 2 — Intent Router

Classify whether a question needs table data, text data, or both.

In [ ]:
router_prompt = ChatPromptTemplate.from_template(
    """You are a query router. Classify the user's question into one category.

Available data:
- TABLE: financial_data (quarterly revenue, expenses, employees, churn, NPS)
- TABLE: product_data (product names, launch years, MAU, revenue, satisfaction)
- TEXT: Company reports about strategy, growth, people, products (qualitative)

Categories:
- TABLE: questions about specific numbers, comparisons, rankings, aggregations
- TEXT: questions about strategy, reasons, explanations, opinions, narratives
- BOTH: questions that need numbers AND qualitative context

Question: {question}

Respond with ONLY one word: TABLE, TEXT, or BOTH"""
)


def route_question(question: str) -> str:
    chain = router_prompt | llm | StrOutputParser()
    result = chain.invoke({"question": question}).strip().upper()
    # Extract just the category
    for cat in ["BOTH", "TABLE", "TEXT"]:
        if cat in result:
            return cat
    return "TEXT"  # default fallback


# Test routing
test_questions = [
    "What was the revenue in Q3 2023?",
    "Why is churn rate declining?",
    "Which product has the highest revenue and why is it successful?",
    "How many employees were there in Q4 2023?",
    "What is the company's growth strategy?",
]

for q in test_questions:
    route = route_question(q)
    print(f"  [{route:5s}] {q}")

## Step 3 — Table Query Engine

The LLM writes a pandas query, we execute it safely, and return the result.

In [ ]:
table_prompt = ChatPromptTemplate.from_template(
    """Write a pandas expression to answer this question.
You have two DataFrames available:

financial_data columns: {fin_cols}
Sample row: {fin_sample}

product_data columns: {prod_cols}
Sample row: {prod_sample}

Question: {question}

Respond with ONLY valid Python pandas code (one expression, no imports).
Use financial_data or product_data as the DataFrame name.

Pandas expression:"""
)


# Safe subset of allowed names for eval
SAFE_NAMES = {
    "financial_data": financial_data,
    "product_data": product_data,
    "pd": pd,
}


def query_table(question: str) -> str:
    """Generate and execute a pandas query."""
    chain = table_prompt | llm | StrOutputParser()
    code = chain.invoke({
        "question": question,
        "fin_cols": list(financial_data.columns),
        "fin_sample": financial_data.iloc[0].to_dict(),
        "prod_cols": list(product_data.columns),
        "prod_sample": product_data.iloc[0].to_dict(),
    })

    # Clean code — get last line if multi-line, strip markdown
    code = code.strip().strip('`').replace('python\n', '')
    lines = [l.strip() for l in code.split('\n') if l.strip() and not l.strip().startswith('#')]
    code = lines[-1] if lines else code

    print(f"  🐍 Generated: {code}")

    # Security: only allow access to our DataFrames and pandas
    try:
        result = eval(code, {"__builtins__": {}}, SAFE_NAMES)
        return f"Query: {code}\nResult:\n{result}"
    except Exception as e:
        return f"Query failed ({code}): {e}. Raw data available:\n{financial_data.to_string()}\n\n{product_data.to_string()}"


# Test table queries
print(query_table("What was revenue in Q3 2023?"))
print("\n" + "-"*40)
print(query_table("Which product has the most monthly active users?"))

## Step 4 — Unified QA Pipeline

In [ ]:
answer_prompt = ChatPromptTemplate.from_template(
    """Answer the question using the provided evidence.
If the evidence includes table results, cite specific numbers.
If it includes text, reference the qualitative context.

Evidence:
{evidence}

Question: {question}

Answer:"""
)


def table_text_qa(question: str) -> str:
    """Full pipeline: route → retrieve → answer."""
    route = route_question(question)
    print(f"\n❓ {question}")
    print(f"🔀 Route → {route}\n")

    evidence_parts = []

    if route in ("TABLE", "BOTH"):
        table_result = query_table(question)
        evidence_parts.append(f"[TABLE DATA]\n{table_result}")

    if route in ("TEXT", "BOTH"):
        text_docs = vectorstore.similarity_search(question, k=3)
        text_evidence = "\n\n".join(f"[{d.metadata.get('source', 'doc')}] {d.page_content}" for d in text_docs)
        evidence_parts.append(f"[TEXT DATA]\n{text_evidence}")

    evidence = "\n\n---\n\n".join(evidence_parts)

    chain = answer_prompt | llm | StrOutputParser()
    answer = chain.invoke({"evidence": evidence, "question": question})

    print(f"\n📝 Answer: {answer}")
    return answer

In [ ]:
# TABLE question — needs numbers
_ = table_text_qa("What was the total revenue across all quarters of 2023?")

In [ ]:
# TEXT question — needs explanation
_ = table_text_qa("Why has the churn rate been declining?")

In [ ]:
# BOTH question — needs numbers + context
_ = table_text_qa("Which product has the highest revenue and what explains its success?")

In [ ]:
# BOTH question — comparing table data with strategic context
_ = table_text_qa("How has employee count changed, and what key hires were made?")

## 🧠 Key Concepts Recap

| Component | What It Does |
|-----------|-------------|
| **Intent Router** | Classifies question → TABLE, TEXT, or BOTH |
| **Table Engine** | LLM generates pandas code, executed safely |
| **Text Retriever** | ChromaDB vector search for qualitative context |
| **Evidence Merger** | Combines both types into unified context |

## Why Table + Text Matters

- **Tables**: Precise numbers, aggregations, comparisons, rankings
- **Text**: Reasons why, strategic context, narratives, explanations
- **Real questions often need both** — "Revenue grew 20% — why?"

## 🔧 Extensions

- Load real CSVs with `pd.read_csv()` and real PDFs for text
- Add SQL database support alongside DataFrames
- Use structured output parsing for more reliable pandas code generation
- Add a chart generator to visualize table query results